# Banking Intent Classification Challenge

**Release:** May 29, 2026  
**Deadline:** June 6, 2026  
**Recommended environment:** Kaggle/Google Colab with GPU runtime

In this competition you will build a text classifier for 77 banking customer-support intents. This notebook is a starting template: it contains the common setup, private Kaggle competition download instructions, sanity checks, and submission validation. It does **not** contain a working baseline model. You are expected to design, train, compare, and analyze your own models.

To get the maximum score, your submission should include:
- one valid Kaggle submission
- one reproducible notebook or code file
- at least **two meaningful experiments on model/backbone choices**
- at least **three meaningful experiments on the data or training setup**
- a small experiment table with validation scores
- error analysis on validation mistakes
- final `submission.csv` generated by your code

## Model Restrictions

The goal of this competition is to practice text classification, validation,
fine-tuning, and error analysis, not to win by using the largest available model.

For the final submitted model:

- You may use classical ML models, neural networks trained from scratch, and
  pretrained encoder-style language models.
- The final model must have at most 250M parameters.
- Closed-source API models, chat models, and instruction-tuned LLMs may not be
  used to generate final test predictions.
- You may use AI assistants for coding and debugging, but not as labelers for
  the hidden test set.
- You may not use external labeled datasets for this task.
- If you use an ensemble, every model in the ensemble must satisfy the same
  restrictions, and the ensemble must be described in your report.
  
## Important rules:
- Use only the provided training data for training.
- Do not manually label test queries.
- Do not search for exact test queries online to recover labels.
- Do not use original Banking77 test labels or any hidden-label source.
- AI tools are allowed, but you must understand and be able to explain your final solution.

Leaderboard score is only part of the grade. Your comparison, ablation, error analysis, and reproducibility also matter.

Grading grid:

| Component | Weight |
|---|---:|
| Private leaderboard score | 50% |
| Model comparison / ablation | 30% |
| Error analysis | 10% |
| Reproducible notebook/code | 10% |

Private leaderboard thresholds:

| Private score | Leaderboard points |
|---:|---:|
| Valid submission below 0.70 | up to 10 / 60 |
| `>= 0.70` | 20 / 60 |
| `>= 0.88` | 30 / 60 |
| `>= 0.88` | 40 / 60 |
| `>= 0.92` | 50 / 60 |
| `>= 0.94` | 60 / 60 |
| `>= 0.97` | bonus consideration |

## 0. Setup

The setup below is provided because it should be almost identical for everyone. You may modify paths and the competition slug if needed.

In [ ]:
# If you are running in Colab, this installs the Kaggle API client.
# You can skip this cell if Kaggle is already installed.
!pip -q install kaggle

In [ ]:
from pathlib import Path
import os
import random
import shutil

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Download the Competition Data

This is a private Kaggle competition. The Kaggle API works only after you:

1. Log in to Kaggle.
2. Accept the private competition invitation in the browser.
3. Upload your own `kaggle.json` API token.
4. Use the correct competition slug from the competition URL.

The invitation token is not the same thing as the competition slug. If you see `No zip files were downloaded`, usually one of these steps is missing.

In [ ]:
# Upload kaggle.json in Colab if needed.
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

if not (kaggle_dir / 'kaggle.json').exists():
    if not IN_COLAB:
        raise FileNotFoundError(
            'Place kaggle.json at ~/.kaggle/kaggle.json, or run this notebook in Colab.'
        )
    uploaded = files.upload()
    if 'kaggle.json' not in uploaded:
        raise RuntimeError('Please upload kaggle.json from your Kaggle account settings.')
    shutil.move('kaggle.json', kaggle_dir / 'kaggle.json')
    os.chmod(kaggle_dir / 'kaggle.json', 0o600)

print('Kaggle API token is ready.')

In [ ]:
COMPETITION_SLUG = 'harbour-space-banking-intent-classification'  # TODO: change if needed
DATA_DIR = Path('/content/banking77_data') if IN_COLAB else Path('banking77_data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c {COMPETITION_SLUG} -p {DATA_DIR}

zip_files = sorted(DATA_DIR.glob('*.zip'))
if not zip_files:
    raise RuntimeError('No zip files were downloaded. Check the competition slug and API access.')

for zip_path in zip_files:
    shutil.unpack_archive(str(zip_path), str(DATA_DIR))

print('Downloaded files:')
for path in sorted(DATA_DIR.iterdir()):
    print('-', path.name)

## 2. Load Metadata and Check the Files

This code only reads metadata and checks that the dataset is present. It does not solve the modeling task.

In [ ]:
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'
LABEL_MAP_CSV = DATA_DIR / 'label_map.csv'
SAMPLE_SUBMISSION_CSV = DATA_DIR / 'sample_submission.csv'

required_paths = [TRAIN_CSV, TEST_CSV, LABEL_MAP_CSV, SAMPLE_SUBMISSION_CSV]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing expected files:\n' + '\n'.join(missing))

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
label_map = pd.read_csv(LABEL_MAP_CSV)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_CSV)

print('train_df:', train_df.shape)
print('test_df:', test_df.shape)
print('label_map:', label_map.shape)
print('sample_submission:', sample_submission.shape)

display(train_df.head())
display(test_df.head())
display(label_map.head())

In [ ]:
label_names = sorted(train_df['label'].unique().tolist())
label_to_id = {label: idx for idx, label in enumerate(label_names)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print('Number of classes:', len(label_names))
print(label_names[:20], '...')

class_counts = train_df['label'].value_counts().sort_index()
display(class_counts.to_frame('count'))

assert len(label_names) == 77, 'Expected 77 intent classes.'
assert set(sample_submission.columns) == {'id', 'label'}, sample_submission.columns.tolist()
assert len(sample_submission) == len(test_df), 'sample_submission and test.csv should have the same number of rows.'
print('Basic checks passed.')

## 3. Inspect Text Examples

Use this section to understand the data before modeling. You may extend it with your own exploratory analysis.

In [ ]:
def show_text_examples(df, n=12, seed=SEED):
    sample = df.sample(n=min(n, len(df)), random_state=seed).reset_index(drop=True)
    for i, row in sample.iterrows():
        label = row.get('label', '<hidden>')
        print(f'[{i:02d}] label={label}')
        print(row['text'])
        print('-' * 80)

show_text_examples(train_df, n=12)

In [ ]:
train_df['text_length_words'] = train_df['text'].astype(str).str.split().str.len()
print(train_df['text_length_words'].describe())

plt.figure(figsize=(8, 4))
train_df['text_length_words'].hist(bins=30)
plt.xlabel('words per query')
plt.ylabel('count')
plt.title('Training query length distribution')
plt.show()

## 4. Create Your Validation Split

Do not tune your model only on the public leaderboard. Create your own validation split from `train.csv`.

Requirements:
- Use a stratified split so every class is represented.
- Keep the split fixed across experiments.
- Report validation accuracy for every experiment in your table.

Suggested variables to create:
- `train_part`
- `val_part`

You may use `sklearn.model_selection.train_test_split`, but the implementation is up to you.

In [ ]:
# TODO: create a stratified train/validation split.
#
# Hints:
# from sklearn.model_selection import train_test_split
# train_part, val_part = train_test_split(..., stratify=train_df['label'], random_state=SEED)

raise NotImplementedError('Create your train/validation split here.')

## 5. Text Preprocessing / Tokenization

Design the preprocessing appropriate for your model.

Requirements:
- Apply the same preprocessing to train, validation, and test.
- Avoid using hidden test labels or external label sources.
- Keep preprocessing choices documented in your experiment table.

Things to experiment with:
- lowercase vs original case;
- punctuation handling;
- TF-IDF features;
- vocabulary size;
- maximum sequence length;
- tokenizer/model choice for Transformers.

In [ ]:
# TODO: implement preprocessing, vectorization, tokenization, or datasets.
#
# Suggested objects depend on your approach. For example:
# - vectorizer for a classical model
# - vocabulary and Dataset for a neural model
# - tokenizer and encoded datasets for a Transformer model

raise NotImplementedError('Implement your text preprocessing or tokenization here.')

## 6. Model Experiments

You need at least **two meaningful experiments on model/backbone choices**.

Examples:
- classical text classifier;
- neural classifier trained from scratch;
- frozen pretrained embeddings;
- fine-tuned Transformer;
- different pretrained language models.

Do not just change a variable name and call it a new experiment. Each experiment should test a real idea.

In [ ]:
# TODO: define your model or model-building function.
#
# Suggested pattern:
# def build_model(...):
#     ...
#     return model

raise NotImplementedError('Define your model experiment here.')

## 7. Training and Validation

Write or reuse a training loop or estimator workflow. Track validation metrics.

Requirements:
- validation accuracy;
- clear model selection rule;
- enough logging to compare experiments fairly.

Suggested variables to create:
- `history` as a pandas DataFrame, if using a neural training loop;
- `best_model_path` or final fitted estimator;
- `best_val_acc`.

In [ ]:
# TODO: implement training and validation.
#
# Suggested skeleton for neural models:
# for epoch in range(num_epochs):
#     # train one epoch
#     # validate
#     # save best checkpoint
#     pass

raise NotImplementedError('Implement training and validation here.')

## 8. Experiment Log

Fill this table as you work. Add rows for every serious experiment.

| Experiment | Model/backbone | Data/training change | Max length / features | LR / C | Val acc | Public LB | Notes |
|---|---|---|---:|---:|---:|---:|---|
| 1 |  |  |  |  |  |  |  |
| 2 |  |  |  |  |  |  |  |
| 3 |  |  |  |  |  |  |  |
| 4 |  |  |  |  |  |  |  |
| 5 |  |  |  |  |  |  |  |

At minimum, you need:
- two meaningful model/backbone experiments;
- three meaningful data or training setup experiments.

## 9. Error Analysis

Use your validation set for error analysis.

Questions to answer:
- Which intent classes are most often confused?
- Show several validation queries your model classified incorrectly.
- Are mistakes caused by semantically similar intents, short queries, vague wording, or missing context?
- Which experiment changed the error pattern the most?

You may implement a confusion matrix and inspect misclassified examples.

In [ ]:
# TODO: collect validation predictions and perform error analysis.
#
# Ideas:
# - confusion matrix
# - most confused intent pairs
# - table of misclassified validation queries

raise NotImplementedError('Add your error analysis here.')

## 10. Generate a Submission

After you choose your final model, predict labels for every row in `test.csv` and create `submission.csv`.

The helper below checks the file format, but it does not create predictions for you.

In [ ]:
def validate_submission(submission_df, test_df, valid_labels):
    required_columns = ['id', 'label']
    assert list(submission_df.columns) == required_columns, submission_df.columns.tolist()
    assert len(submission_df) == len(test_df), 'Wrong number of rows.'
    assert submission_df['id'].is_unique, 'Submission IDs must be unique.'
    assert set(submission_df['id']) == set(test_df['id']), 'Submission IDs must match test.csv.'
    bad_labels = sorted(set(submission_df['label']) - set(valid_labels))
    assert not bad_labels, f'Invalid labels found: {bad_labels[:10]}'
    assert not submission_df['label'].isna().any(), 'Missing labels found.'
    print('Submission format looks valid.')

# TODO: replace this with predictions from your trained model.
# final_predictions should be a list/array of labels, one per row in test_df.

final_predictions = None

if final_predictions is None:
    raise NotImplementedError('Create final_predictions using your trained model.')

submission = pd.DataFrame({
    'id': test_df['id'],
    'label': final_predictions,
})

validate_submission(submission, test_df, label_names)
submission.to_csv('submission.csv', index=False)
display(submission.head())
print('Saved submission.csv')

## 11. Final Checklist

Before submitting, check that:

- `submission.csv` has exactly the columns `id,label`;
- the number of rows equals `test.csv`;
- all predicted labels are valid class names;
- your notebook can reproduce the submission file;
- your experiment table contains at least five meaningful experiments total;
- your error analysis uses validation data, not the hidden test set;
- you can explain your final model and why you chose it.